In [47]:
from collections import defaultdict
from datetime import date
import pandas as pd
import csv
import os

In [48]:
# read the pcr spreadsheets containng PCR plate code, sample codes, and barcodes
pcr_sample_status = pd.read_table('/mnt/c/Users/mftor/Documents/saproxDNA/species_info_seq5/PCR_to_species_20260602.txt')
pcr_sample_status

,PCR ID,Sample well,Sample ID,Success tier,Nanodrop concentration,260/280 ratio,260/230 ratio,Forward tag name,Reverse tag name,Forward tag sequence,Reverse tag sequence
0,single_tube,individual tube,positive,good,NaN,NaN,NaN,bc1007,bc1122rc,GGTAGTCTGTATCTCTATGTGRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
1,PCR_Sx_D3_20260508,A1,A-1,low,9.9,2.26,-1.64,bc1001,bc1122rc,GGTAGCACATATCAGAGTGCGRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
2,PCR_Sx_D3_20260508,A10,102,good,8.1,3.11,-0.76,bc1010,bc1122rc,GGTAGACGCGCTATCTCAGAGRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
3,PCR_Sx_D3_20260508,A11,82,low,11.5,2.03,-8.01,bc1011,bc1122rc,GGTAGCTATACGTATATCTATRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
4,PCR_Sx_D3_20260508,A12,90,good,5.5,2.9,-0.78,bc1012,bc1122rc,GGTAGACACTAGATCGCGTGTRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
...,...,...,...,...,...,...,...,...,...,...,...
2142,PCRSxPh_1__20260312,H5,SxPh_s41,low,1.8,2.15,0.74,bc1089,bc1122rc,GGTAGCGCGCGAGTATCTCGTRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
2143,PCRSxPh_1__20260312,H6,SxPh_s33,low,4.5,1.55,0.59,bc1090,bc1122rc,GGTAGAGCACACATATAGCGCRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
2144,PCRSxPh_1__20260312,H7,SxPh_s25,low,4.2,1.8,0.48,bc1091,bc1122rc,GGTAGGTATATATATACGTCTRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA
2145,PCRSxPh_1__20260312,H8,SxPh_s17,low,5.1,1.77,0.57,bc1092,bc1122rc,GGTAGTCTCACGAGAGCGCACRKTCAACMAATCATAAAGATATTGG,GGTAGGCGTGTGAGAGTCTGCTAAACTTCWGGRTGWCCAAAAAATCA


In [49]:
# homogenise the sample spreadsheets sent by taxonomists after loads of manual editing
# VP masalai doesnt even have unique codes that follow the syntax!!! how the hell to connect them to PCR plate wells?
al_coleoptera = pd.read_csv('./AL_SaproxDNA_Coleoptera_list_20260528.txt', sep = '\t')
ig_subdiptera = pd.read_csv('./IG_Rusys_DNR_SaproX_all_tabs_cat_20260602.txt', sep = '\t')
vp_masalai = pd.read_csv('./VP_Podenienes_masalai_20260602.txt', sep = '\t')
vp_maziaupuses = pd.read_csv('./VP_tarpiniai_duomenys_maziau_puses_musiu_20260602.txt', sep = '\t')

spp_provided = [al_coleoptera, ig_subdiptera, vp_masalai, vp_maziaupuses]

# for db in spp_provided:
#     print(db.columns)

# for db in spp_provided:
#     print(db.species.head())

# to create dictionaries linking saproxdna codes to species identified
# each saproxdna code must have a unique species
# but one species key can have multiple saproxdna codes if multiple specimens of the same species was sequenced
# global dictionary key: saproxdna code; value: species identity based on morphology

sxdcode_spp  = {}
for db in spp_provided:
    for code, species in zip(db['sxdcode'], db['species']):
        # safe check for accidentally duplicated saproxdna codes
        if code in sxdcode_spp and sxdcode_spp[code] != species:
            print(f'Conflict: {code} → {sxdcode_spp[code]} / {species}')
        # if there's no duplicated codes, add it to the global dictionary
        sxdcode_spp[code] = species

Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: nan → nan / nan
Conflict: na

In [50]:
# exporting the conflicts
spp_provided = [al_coleoptera, ig_subdiptera, vp_masalai, vp_maziaupuses]

spp_provided_names = [('al_coleoptera', al_coleoptera),
    ('ig_subdiptera', ig_subdiptera),
    ('vp_masalai', vp_masalai),
    ('vp_maziaupuses', vp_maziaupuses)]

# global dictionary
sxdcode_spp = {}
# dataset from which the code comes from
code_source = {}

with open('sxdcode_conflicts.tsv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')

    writer.writerow(['sxdcode',
        'first_species',
        'first_database',
        'second_species',
        'second_database'])

    for database_name, db in spp_provided_names:
        for code, species in zip(db['sxdcode'], db['species']):

            if code in sxdcode_spp and sxdcode_spp[code] != species:
                writer.writerow([
                    code,
                    sxdcode_spp[code],
                    code_source[code],
                    species,
                    database_name,
                ])

            else:
                # keep the first code:species pair observed
                sxdcode_spp[code] = species
                code_source[code] = database_name

In [51]:
version = 1
generation_date = date.today().strftime("%Y%m%d")

output_file = (
    f'saproxdna_tospecies_{generation_date}_v{version}.tsv'
)

with open(output_file, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')

    writer.writerow(['sxdcode', 'species'])

    for code, species in sxdcode_spp.items():
        writer.writerow([code, species])

print(f'Exported {len(sxdcode_spp)} records to {output_file}')

Exported 2367 records to saproxdna_tospecies_20260630_v1.tsv
